# OmniVoice 기초

- [k2-fsa/OmniVoice](https://github.com/k2-fsa/OmniVoice) 는 600 + 언어를 지원하는 diffusion 기반 TTS 모델. 
- 첫 호출 시 가중치를 자동으로 다운로드 (수 분, 한 번만).


## 1. 환경 확인

GPU 가 잡혀 있는지, 메모리는 얼마인지 먼저 봅니다.

In [1]:
import torch
print("PyTorch 버전:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

PyTorch 버전: 2.10.0+cpu
CUDA available: False


## 2. 모델 로드

첫 호출 시 가중치가 `~/.cache/huggingface/` 아래로 다운로드됩니다. 처음에는 시간이 좀 걸리고, 두 번째부터는 즉시 로드.

- Hugging Face 토큰 에러 발생시 아래 코드 실행

In [2]:
# import os

# # Hugging Face Hub가 내 PC에 저장된 로그인 토큰을 자동으로 사용하지 말고, 공개 모델이면 그냥 익명으로 다운로드해라.
# os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
# try:
#     import huggingface_hub.constants as hf_constants
#     hf_constants.HF_HUB_DISABLE_IMPLICIT_TOKEN = True
# except Exception:
#     pass

In [3]:
# 라이브러리 다운로드
%pip install omnivoice

Note: you may need to restart the kernel to use updated packages.


In [4]:
from omnivoice import OmniVoice

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

c:\Users\playdata2\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).
W0709 11:26:54.915000 36064 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [5]:
import soundfile as sf
import torch
from IPython.display import Audio
import os
os.makedirs("samples", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map=device,
    dtype=dtype
)
print("모델 로드 완료. device:", device)

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]c:\Users\playdata2\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\playdata2\.cache\huggingface\hub\models--k2-fsa--OmniVoice. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 527/527 [00:00<00:00, 1175.

모델 로드 완료. device: cpu


## 3. 첫 합성, 영어 한 문장

모델은 텍스트를 받아 NumPy 배열 (`shape=(T,)`, 24kHz) 을 돌려줍니다.

In [6]:
import soundfile as sf
import os
os.makedirs("outputs", exist_ok=True)

audio = model.generate(
    text="Hello from OmniVoice. This is a test."
)
print("audio shape:", audio[0].shape, "sample rate: 24000")

# audio[0] : 모델이 생성한 음성 파형 데이터
# 24000 : sampling rate. 즉, 1초의 소리를 표현하기 위해 24,000개의 숫자 샘플을 사용한다는 뜻.
sf.write("./outputs/hello_en.wav", audio[0], 24000)
print("저장 완료")

audio shape: (57600,) sample rate: 24000
저장 완료


## 4. 노트북 안에서 바로 재생

`IPython.display.Audio` 로 셀 안에서 바로 재생 가능.

In [7]:
from IPython.display import Audio
Audio("./outputs/hello_en.wav")

## 5. 여러 문장 한 번에

`model.generate` 는 텍스트 하나를 받지만, 루프 돌려서 여러 wav 를 만들 수 있습니다.

In [8]:
phrases = [
    "Welcome to SK networks family ai camp.",
    "Today, we learn voice synthesis.",
    "Press enter to continue.",
]

for i, text in enumerate(phrases):
    audio = model.generate(text=text)
    path = f"./outputs/phrase_{i:02d}.wav"
    sf.write(path, audio[0], 24000)
    print(f"  saved: {path}")

from IPython.display import Audio
Audio("./outputs/phrase_02.wav")

  saved: ./outputs/phrase_00.wav
  saved: ./outputs/phrase_01.wav
  saved: ./outputs/phrase_02.wav
